In [1]:
# Importação da base de dados
import os
import kagglehub

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score,
    ConfusionMatrixDisplay
)

# Desbalanceamento
from imblearn.over_sampling import SMOTE

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Configurações de visualização
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Imports realizados com sucesso!')

Imports realizados com sucesso!


In [2]:
# Download do dataset
path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
print("Path to dataset files:", path)

# Localizar o arquivo CSV dentro da pasta baixada
files = os.listdir(path)
print("Arquivos disponíveis:", files)

# Carregar o CSV
df = pd.read_csv(os.path.join(path, files[0]))

print(f'Shape: {df.shape}')
df.head()

100%|██████████| 0.98M/0.98M [00:01<00:00, 1.02MB/s]

Extracting files...


Path to dataset files: C:\Users\Felipe\.cache\kagglehub\datasets\uciml\default-of-credit-card-clients-dataset\versions\1
Arquivos disponíveis: ['UCI_Credit_Card.csv']
Shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [3]:
# Remover coluna identificadora
df.drop(columns=['ID'], inplace=True)

# Padronizar nome da variável-alvo
df.rename(columns={'default.payment.next.month': 'default'}, inplace=True)

print('Colunas após limpeza:')
print(df.columns.tolist())

Colunas após limpeza:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default']


In [4]:
print('=== Informações gerais ===')
df.info()

print('\n=== Valores ausentes por coluna ===')
print(df.isnull().sum())

=== Informações gerais ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   LIMIT_BAL  30000 non-null  float64
 1   SEX        30000 non-null  int64  
 2   EDUCATION  30000 non-null  int64  
 3   MARRIAGE   30000 non-null  int64  
 4   AGE        30000 non-null  int64  
 5   PAY_0      30000 non-null  int64  
 6   PAY_2      30000 non-null  int64  
 7   PAY_3      30000 non-null  int64  
 8   PAY_4      30000 non-null  int64  
 9   PAY_5      30000 non-null  int64  
 10  PAY_6      30000 non-null  int64  
 11  BILL_AMT1  30000 non-null  float64
 12  BILL_AMT2  30000 non-null  float64
 13  BILL_AMT3  30000 non-null  float64
 14  BILL_AMT4  30000 non-null  float64
 15  BILL_AMT5  30000 non-null  float64
 16  BILL_AMT6  30000 non-null  float64
 17  PAY_AMT1   30000 non-null  float64
 18  PAY_AMT2   30000 non-null  float64
 19  PAY_AMT3   30000 no

In [5]:
# Estatísticas descritivas
df.describe().T

,count,mean,std,min,25%,50%,75%,max
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0
PAY_5,30000.0,-0.266200,1.133187,-2.0,-1.00,0.0,0.00,8.0


In [6]:
print('Antes do tratamento:')
print('EDUCATION:', df['EDUCATION'].value_counts().sort_index().to_dict())
print('MARRIAGE: ', df['MARRIAGE'].value_counts().sort_index().to_dict())

# Agrupar categorias não documentadas em 'outros' (4 e 3 respectivamente)
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE']  = df['MARRIAGE'].replace({0: 3})

print('\nApós tratamento:')
print('EDUCATION:', df['EDUCATION'].value_counts().sort_index().to_dict())
print('MARRIAGE: ', df['MARRIAGE'].value_counts().sort_index().to_dict())

Antes do tratamento:
EDUCATION: {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
MARRIAGE:  {0: 54, 1: 13659, 2: 15964, 3: 323}

Após tratamento:
EDUCATION: {1: 10585, 2: 14030, 3: 4917, 4: 468}
MARRIAGE:  {1: 13659, 2: 15964, 3: 377}


In [7]:
X = df.drop(columns=['default'])
y = df['default']

# Divisão treino+val / teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)

# Divisão treino / validação
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.15 / 0.85), random_state=SEED, stratify=y_temp
)

print(f'Treino:    {X_train.shape[0]:>6} registros ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Validação: {X_val.shape[0]:>6} registros ({X_val.shape[0]/len(df)*100:.1f}%)')
print(f'Teste:     {X_test.shape[0]:>6} registros ({X_test.shape[0]/len(df)*100:.1f}%)')
print(f'\nProporção de default — treino: {y_train.mean():.3f} | teste: {y_test.mean():.3f}')

Treino:     21000 registros (70.0%)
Validação:   4500 registros (15.0%)
Teste:       4500 registros (15.0%)

Proporção de default — treino: 0.221 | teste: 0.221


In [8]:
# Normalização — fit apenas no treino
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Normalização concluída.')
print(f'Média no treino (esperado ~0): {X_train_sc.mean():.4f}')
print(f'Desvio padrão no treino (esperado ~1): {X_train_sc.std():.4f}')

Normalização concluída.
Média no treino (esperado ~0): -0.0000
Desvio padrão no treino (esperado ~1): 1.0000
